# World Model Pipeline — Full Experiment Walkthrough

This notebook walks through the **complete world model pipeline** for our DS-GA 3001 project, including:

1. **Data Loading** — Loading robot trajectories from Hugging Face Hub
2. **Architecture** — The modular world model (vision encoder + proprioception + fusion + dynamics)
3. **Training Demo** — Running the fastest config (CNN + MLP) for 5 epochs on GPU
4. **Full Experiment Results** — Loading results from all 8 SLURM training runs (100 epochs each)
5. **Comparison & Visualization** — Training curves, ADE/FDE tables, and plots
6. **MPC Inference** — Using a trained model for planning

### Experiment Grid (2 × 2 × 2 = 8 runs)

| Dimension | Options |
|-----------|---------|
| **Vision Backbone** | CNN (lightweight) · DINOv2-ViT-B/14 (pretrained, frozen) |
| **Dynamics Model** | MLP (feedforward) · RSSM (recurrent state-space) |
| **Dataset** | `collect1` (filtered) · `collect1_unfiltered` (all trajectories) |

All runs: 100 epochs, AdamW (lr=3e-4), horizon=16, frozen backbone for DINOv2.

**Actions** are not a separate motor command — they are the **velocity slice of odometry** (`odom[7:13]` = lin_vel[3] + ang_vel[3], 6-dim).

In [1]:
%pip install pandas datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext autoreload
%autoreload 2
import os, sys, json, glob, time
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader

# Project paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
SCRATCH = f"/scratch/{os.environ['USER']}"
RESULTS_DIR = os.path.join(SCRATCH, "wm_experiments_v3")  # v3: velocity actions (6-dim)

from world_model.data import HFWindowDataset, collate_hf
from world_model.data.normalization import NormStats
from world_model.architecture import WorldModel
from world_model.engine.trainer import TrainConfig, rollout_loss
from world_model.mpc import cem_plan, CEMConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Experiments found: {sorted(os.listdir(RESULTS_DIR)) if os.path.exists(RESULTS_DIR) else 'N/A (training in progress)'}")

ModuleNotFoundError: No module named 'torch'

## 1. Data Loading

We use two versions of the dataset hosted on Hugging Face:
- **`lquan9/collect1`** — Filtered trajectories (825 train / 207 val windows)
- **`lquan9/collect1_unfiltered`** — All trajectories including noisy ones (1444 train / 362 val)

Each window contains synchronized:
- **RGB images** (224×224 for DINOv2, 128×128 for CNN)
- **Odometry** (13-dim: pos[3] + quat[4] + lin_vel[3] + ang_vel[3])
- **IMU** (10-dim: accelerometer + gyroscope)
- **Actions** = `odom[:, 7:13]` (6-dim: lin_vel + ang_vel — not a separate sensor, just the velocity slice of odometry)

### Load & Inspect the Dataset

In [ ]:
hf_repo = "lquan9/collect1"
print(f"Loading {hf_repo}...")
ds_hf = HFWindowDataset(hf_repo, img_size=128)  # 128 for CNN (faster loading)

# Override legacy action columns -> use velocity from odom (6-dim)
ds_hf.force_velocity_actions()

print(f"Total windows (episodes): {len(ds_hf)}")
print(f"Total frames in dataset:  {len(ds_hf._ds)}")
print(f"Columns: {ds_hf._ds.column_names}")
print(f"Dimensions: Odom={ds_hf.D_odom}, IMU={ds_hf.D_imu}, Act={ds_hf.D_act} (velocity: lin_vel + ang_vel)")

# Split info (matches training script: val = episode_id % 5 == 0)
all_ids = ds_hf._window_ids
val_ids = [eid for eid in all_ids if eid % 5 == 0]
train_ids = [eid for eid in all_ids if eid % 5 != 0]
print(f"\nTrain/Val split: {len(train_ids)} train, {len(val_ids)} val (eid % 5 == 0 -> val)")

### What does one trajectory look like?

Each episode is a window of synchronized sensor readings. Let's visualize all the modalities for one episode side by side.

In [ ]:
window = ds_hf[0]
rgb, odom, imu, act = window["rgb"], window["odom"], window["imu"], window["action"]
T = rgb.shape[0]
timesteps = np.arange(T) * 0.2  # 5 Hz

fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(5, 5, hspace=0.5, wspace=0.3)

# Row 1: Camera images across the trajectory
for i, idx in enumerate([0, T//4, T//2, 3*T//4, T-1]):
    ax = fig.add_subplot(gs[0, i])
    ax.imshow(rgb[idx].transpose(1, 2, 0))
    ax.set_title(f"t={timesteps[idx]:.1f}s", fontsize=10)
    ax.axis('off')
fig.text(0.08, 0.86, "Camera\n(RGB)", ha='center', va='center', fontsize=11, fontweight='bold')

# Row 2: Odometry — position (x, y, z) and orientation
ax_pos = fig.add_subplot(gs[1, :3])
ax_pos.plot(timesteps, odom[:, 0], label='x')
ax_pos.plot(timesteps, odom[:, 1], label='y')
ax_pos.plot(timesteps, odom[:, 2], label='z')
ax_pos.set_ylabel('Position (m)')
ax_pos.set_title('Odometry: Position (odom[0:3])')
ax_pos.legend(fontsize=8)
ax_pos.grid(True, alpha=0.3)

ax_ori = fig.add_subplot(gs[1, 3:])
ax_ori.plot(timesteps, odom[:, 3], label='qx')
ax_ori.plot(timesteps, odom[:, 4], label='qy')
ax_ori.plot(timesteps, odom[:, 5], label='qz')
ax_ori.plot(timesteps, odom[:, 6], label='qw')
ax_ori.set_ylabel('Quaternion')
ax_ori.set_title('Odometry: Orientation (odom[3:7])')
ax_ori.legend(fontsize=8)
ax_ori.grid(True, alpha=0.3)

# Row 3: Odometry velocities = Actions
ax_linvel = fig.add_subplot(gs[2, :3])
ax_linvel.plot(timesteps, odom[:, 7], label='vx (odom[7])', linewidth=2)
ax_linvel.plot(timesteps, odom[:, 8], label='vy (odom[8])', linewidth=2)
ax_linvel.plot(timesteps, odom[:, 9], label='vz (odom[9])', linewidth=2)
# Overlay action dims to show they match
ax_linvel.plot(timesteps, act[:, 0], '--', label='act[0]', alpha=0.5, color='black')
ax_linvel.plot(timesteps, act[:, 1], '--', label='act[1]', alpha=0.5, color='gray')
ax_linvel.plot(timesteps, act[:, 2], '--', label='act[2]', alpha=0.5, color='silver')
ax_linvel.set_ylabel('Lin Velocity (m/s)')
ax_linvel.set_title('Action = Linear Velocity (odom[7:10]) — dashed = action dims')
ax_linvel.legend(fontsize=7, ncol=2)
ax_linvel.grid(True, alpha=0.3)

ax_angvel = fig.add_subplot(gs[2, 3:])
ax_angvel.plot(timesteps, odom[:, 10], label='wx (odom[10])', linewidth=2)
ax_angvel.plot(timesteps, odom[:, 11], label='wy (odom[11])', linewidth=2)
ax_angvel.plot(timesteps, odom[:, 12], label='wz (odom[12])', linewidth=2)
ax_angvel.plot(timesteps, act[:, 3], '--', label='act[3]', alpha=0.5, color='black')
ax_angvel.plot(timesteps, act[:, 4], '--', label='act[4]', alpha=0.5, color='gray')
ax_angvel.plot(timesteps, act[:, 5], '--', label='act[5]', alpha=0.5, color='silver')
ax_angvel.set_ylabel('Ang Velocity (rad/s)')
ax_angvel.set_title('Action = Angular Velocity (odom[10:13]) — dashed = action dims')
ax_angvel.legend(fontsize=7, ncol=2)
ax_angvel.grid(True, alpha=0.3)

# Row 4: IMU signals
ax_acc = fig.add_subplot(gs[3, :3])
ax_acc.plot(timesteps, imu[:, 0], label='acc_x')
ax_acc.plot(timesteps, imu[:, 1], label='acc_y')
ax_acc.plot(timesteps, imu[:, 2], label='acc_z')
ax_acc.set_ylabel('Accel (m/s^2)')
ax_acc.set_title('IMU: Accelerometer')
ax_acc.legend(fontsize=8)
ax_acc.grid(True, alpha=0.3)

ax_gyro = fig.add_subplot(gs[3, 3:])
if imu.shape[1] > 3:
    ax_gyro.plot(timesteps, imu[:, 3], label='gyro_x')
    ax_gyro.plot(timesteps, imu[:, 4], label='gyro_y')
    ax_gyro.plot(timesteps, imu[:, 5], label='gyro_z')
ax_gyro.set_ylabel('Angular vel (rad/s)')
ax_gyro.set_title('IMU: Gyroscope')
ax_gyro.legend(fontsize=8)
ax_gyro.grid(True, alpha=0.3)

# Row 5: Odom layout diagram
ax_layout = fig.add_subplot(gs[4, :])
ax_layout.axis('off')
layout_text = (
    "Odometry Layout (13-dim):  [ pos_x  pos_y  pos_z | qx  qy  qz  qw | lin_vx  lin_vy  lin_vz | ang_wx  ang_wy  ang_wz ]\n"
    "                             odom[0:3] position    odom[3:7] orient   odom[7:10] lin_vel        odom[10:13] ang_vel\n"
    "                                                                      \_____________________________________________/\n"
    "                                                                              Action = odom[7:13] (6-dim)"
)
ax_layout.text(0.05, 0.5, layout_text, fontsize=10, family='monospace',
               transform=ax_layout.transAxes, verticalalignment='center',
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle(f"Episode 0 — All Sensor Modalities ({T} frames, {timesteps[-1]:.1f}s)", fontsize=14, y=1.01)
plt.show()

In [ ]:
# Bird's-eye view of several trajectories + altitude profile
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sample 8 trajectories
cmap = plt.cm.tab10
for i in range(min(8, len(ds_hf))):
    w = ds_hf[i]
    o = w["odom"]
    color = cmap(i)
    axes[0].plot(o[:, 0], o[:, 1], color=color, alpha=0.7, label=f"Ep {i}")
    axes[0].scatter(o[0, 0], o[0, 1], color=color, marker='o', s=40, zorder=5)
    axes[0].scatter(o[-1, 0], o[-1, 1], color=color, marker='x', s=60, zorder=5)
    axes[1].plot(o[:, 2], color=color, alpha=0.7)

axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
axes[0].set_title("Bird's-Eye Trajectories (o=start, x=end)")
axes[0].legend(fontsize=7, ncol=2)
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Frame'); axes[1].set_ylabel('Z (m)')
axes[1].set_title('Altitude Profiles')
axes[1].grid(True, alpha=0.3)

# Distribution of trajectory lengths
lengths = [ds_hf[i]["odom"].shape[0] for i in range(min(50, len(ds_hf)))]
axes[2].hist(lengths, bins=20, edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Frames per episode')
axes[2].set_ylabel('Count')
axes[2].set_title('Episode Length Distribution (first 50)')
axes[2].axvline(np.mean(lengths), color='red', linestyle='--', label=f'mean={np.mean(lengths):.0f}')
axes[2].legend()

plt.tight_layout()
plt.show()

## 2. Model Architecture

The `WorldModel` is modular with these components:

```
RGB Image ──> [Vision Encoder] ──> vision_proj ──┐
                                                  ├──> [Fusion] ──> z_t (latent state)
Odom + IMU ──> [ProprioEncoder] ─────────────────┘         │
                                                            ├──> [LatentDynamics] ──> z_{t+1}
                              Action (velocity 6d) ─────────┘         │
                              = odom[7:13]                            ├──> [StateHead] ──> predicted odom
                                                                      └──> [RiskHead]  ──> collision risk
```

**Key design choice:** Actions are **not** a separate motor command signal — they are the **velocity portion of odometry** (`odom[7:13]` = 3 linear + 3 angular velocities, 6-dim). This means the dynamics model predicts next latent state from current state + current velocity.

We compare **4 configurations** by swapping:
- **Vision Encoder**: CNN (4-layer ConvNet, 128px) vs DINOv2-ViT-B/14 (pretrained, frozen, 224px)
- **Dynamics Model**: MLP (feedforward) vs RSSM (recurrent state-space model)

In [ ]:
# Build all 4 model configurations and compare parameter counts
o_dim, i_dim, a_dim = ds_hf.D_odom, ds_hf.D_imu, ds_hf.D_act  # act=6 after override

configs = [
    ("CNN + MLP",  dict(backbone="cnn",    dynamics_type="mlp")),
    ("CNN + RSSM", dict(backbone="cnn",    dynamics_type="rssm")),
    ("DINOv2 + MLP",  dict(backbone="dinov2", dynamics_type="mlp",
                           backbone_kwargs={"model_name": "dinov2_vitb14", "device": str(device)},
                           freeze_backbone=True)),
    ("DINOv2 + RSSM", dict(backbone="dinov2", dynamics_type="rssm",
                            backbone_kwargs={"model_name": "dinov2_vitb14", "device": str(device)},
                            freeze_backbone=True)),
]

print(f"Input dims: odom={o_dim}, imu={i_dim}, act={a_dim} (velocity: lin_vel + ang_vel)\n")
print(f"{'Config':<20} {'Total Params':>14} {'Trainable':>14}")
print("-" * 50)

for name, kwargs in configs:
    m = WorldModel(odom_dim=o_dim, imu_dim=i_dim, act_dim=a_dim, **kwargs)
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name:<20} {total:>14,} {trainable:>14,}")
    del m

# Clean up DINOv2 cache
import gc; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

### What does the encoder see?

Let's pass a single frame through the CNN encoder and visualize:
1. The **CNN feature maps** at each convolutional layer
2. The **latent vectors** (z_vis, z_prop, z_fused) that the model computes
3. The **decoded state and risk** predictions from a single frame

In [ ]:
# Build a CNN+MLP model and inspect what happens to an image
vis_model = WorldModel(odom_dim=o_dim, imu_dim=i_dim, act_dim=a_dim,
                       backbone="cnn", dynamics_type="mlp").to(device)

# Get one frame
sample = ds_hf[0]
rgb_t = torch.from_numpy(sample["rgb"][0:1]).to(device)   # (1,3,H,W)
depth_t = torch.zeros(1, 1, 128, 128, device=device)       # placeholder depth
odom_t = torch.from_numpy(sample["odom"][0:1]).to(device)
imu_t = torch.from_numpy(sample["imu"][0:1]).to(device)

# --- 1. CNN Feature Maps ---
# Hook into each conv layer to capture activations
activations = {}
def get_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach().cpu()
    return hook

hooks = []
for i, layer in enumerate(vis_model.vision.net):
    if isinstance(layer, torch.nn.Conv2d):
        hooks.append(layer.register_forward_hook(get_hook(f"Conv {i//2 + 1}")))

# Forward pass through vision encoder
x_vis = torch.cat([rgb_t, depth_t], dim=1)  # (1,4,128,128)
with torch.no_grad():
    z_vis = vis_model.vision(x_vis)

for h in hooks:
    h.remove()

# Plot feature maps
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

# Original image
axes[0, 0].imshow(sample["rgb"][0].transpose(1, 2, 0))
axes[0, 0].set_title("Input RGB (128x128)")
axes[0, 0].axis('off')

# Feature maps for each conv layer (show first 8 channels as a grid mean)
for col, (name, feat) in enumerate(activations.items()):
    feat_np = feat[0].numpy()  # (C, H, W)
    n_ch = min(8, feat_np.shape[0])
    
    # Show mean activation
    axes[0, col+1].imshow(feat_np.mean(axis=0), cmap='viridis')
    axes[0, col+1].set_title(f"{name}: mean ({feat_np.shape[0]}ch, {feat_np.shape[1]}x{feat_np.shape[2]})")
    axes[0, col+1].axis('off')
    
    # Show first 8 channels as a small grid
    grid = np.zeros((2*feat_np.shape[1], 4*feat_np.shape[2]))
    for ch in range(n_ch):
        r, c = ch // 4, ch % 4
        grid[r*feat_np.shape[1]:(r+1)*feat_np.shape[1],
             c*feat_np.shape[2]:(c+1)*feat_np.shape[2]] = feat_np[ch]
    axes[1, col+1].imshow(grid, cmap='viridis')
    axes[1, col+1].set_title(f"{name}: channels 0-7")
    axes[1, col+1].axis('off')

# --- 2. Latent vectors ---
with torch.no_grad():
    z_prop = vis_model.proprio(odom_t, imu_t)
    z_fused = vis_model.fusion(vis_model.vision_proj(z_vis) if hasattr(vis_model, 'vision_proj') else z_vis, z_prop)
    state_pred = vis_model.state_head(z_fused)
    risk_pred = vis_model.risk_head(z_fused)

# Bar chart of latent dimensions
ax = axes[1, 0]
z_vis_np = z_vis[0].cpu().numpy()
ax.bar(range(len(z_vis_np)), z_vis_np, width=1.0, color='steelblue', alpha=0.7)
ax.set_title(f"z_vis ({len(z_vis_np)}-dim)")
ax.set_xlabel("Dimension")

axes[0, 3].axis('off')
# Summary text
act_sample = sample["action"][0]  # 6-dim velocity
summary = (f"Vision:  z_vis  = {z_vis.shape}\n"
           f"Proprio: z_prop = {z_prop.shape}\n"
           f"Fused:   z      = {z_fused.shape}\n"
           f"State:   {state_pred[0].cpu().numpy().round(3)}\n"
           f"Risk:    {risk_pred[0].item():.4f}\n"
           f"Action:  {act_sample.round(3)} (6d vel)")
axes[0, 3].text(0.1, 0.5, summary, fontsize=10, family='monospace',
                transform=axes[0, 3].transAxes, verticalalignment='center',
                bbox=dict(boxstyle='round', facecolor='lightyellow'))
axes[0, 3].set_title("Encoder Output Summary")

plt.suptitle("Inside the CNN Encoder: Image -> Feature Maps -> Latent Vector", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

del vis_model; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

## 3. Training Demo (CNN + MLP, 5 epochs)

This is the **fastest configuration** (~9s/epoch on A100). We run 5 epochs here as a quick demo.
The full 100-epoch SLURM runs are analyzed in Section 4.

**Training details:**
- Multi-step rollout loss: encode frame at t=0, then roll out dynamics for `horizon=16` steps
- Actions = velocity from odom (6-dim: `odom[7:13]` = lin_vel + ang_vel)
- Loss = MSE on latent states (`loss_z`) + MSE on predicted odom (`loss_state`) + BCE on risk (`loss_risk`)
- AdamW optimizer, lr=3e-4, weight_decay=1e-4, gradient clipping at 1.0

In [ ]:
import torch.optim as optim

# --- Data Setup (matches the SLURM training script exactly) ---
# Compute normalization stats from training episodes only
train_episode_ids = [eid for eid in ds_hf._window_ids if eid % 5 != 0]
val_episode_ids   = [eid for eid in ds_hf._window_ids if eid % 5 == 0]

norm_stats = ds_hf.compute_norm_stats(episode_ids=train_episode_ids)

# Rebuild datasets with normalization + velocity actions
ds_train = HFWindowDataset(hf_repo, img_size=128, norm_stats=norm_stats)
ds_train.force_velocity_actions()
ds_train._window_ids = [eid for eid in ds_train._window_ids if eid in set(train_episode_ids)]
ds_train._window_rows = {eid: ds_train._window_rows[eid] for eid in ds_train._window_ids}

ds_val = HFWindowDataset(hf_repo, img_size=128, norm_stats=norm_stats)
ds_val.force_velocity_actions()
ds_val._window_ids = [eid for eid in ds_val._window_ids if eid in set(val_episode_ids)]
ds_val._window_rows = {eid: ds_val._window_rows[eid] for eid in ds_val._window_ids}

print(f"Train windows: {len(ds_train)}, Val windows: {len(ds_val)}")
print(f"Action dim: {ds_train.D_act} (odom velocity: lin_vel + ang_vel)")

train_loader = DataLoader(ds_train, batch_size=64, shuffle=True, collate_fn=collate_hf,
                          num_workers=4, drop_last=True, pin_memory=True)
val_loader   = DataLoader(ds_val, batch_size=64, shuffle=False, collate_fn=collate_hf,
                          num_workers=4, drop_last=False, pin_memory=True)

# --- Model Setup (act_dim=6 for velocity actions) ---
o_dim, i_dim, a_dim = ds_train.D_odom, ds_train.D_imu, ds_train.D_act

model = WorldModel(
    odom_dim=o_dim, imu_dim=i_dim, act_dim=a_dim,
    backbone="cnn", dynamics_type="mlp"
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
cfg = TrainConfig(device=str(device), lr=3e-4, horizon=16)

print(f"Model: CNN + MLP (act_dim={a_dim})")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

### Training Loop (5 epochs)

In [ ]:
EPOCHS = 5
demo_history = {"train_loss": [], "val_loss": [],
                "train_z": [], "train_state": [], "train_risk": [],
                "val_z": [], "val_state": [], "val_risk": []}

t0 = time.time()
for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    sums = {"loss": 0, "z": 0, "state": 0, "risk": 0}
    n_batches = 0
    for batch in train_loader:
        loss, logs = rollout_loss(model, batch, cfg)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        sums["loss"] += logs["loss"]
        sums["z"] += logs["loss_z"]
        sums["state"] += logs["loss_state"]
        sums["risk"] += logs["loss_risk"]
        n_batches += 1

    for k in sums: sums[k] /= max(n_batches, 1)
    demo_history["train_loss"].append(sums["loss"])
    demo_history["train_z"].append(sums["z"])
    demo_history["train_state"].append(sums["state"])
    demo_history["train_risk"].append(sums["risk"])

    # --- Validate ---
    model.eval()
    v_sums = {"loss": 0, "z": 0, "state": 0, "risk": 0}
    v_steps = 0
    with torch.no_grad():
        for batch in val_loader:
            v_loss, v_logs = rollout_loss(model, batch, cfg)
            v_sums["loss"] += v_logs["loss"]
            v_sums["z"] += v_logs["loss_z"]
            v_sums["state"] += v_logs["loss_state"]
            v_sums["risk"] += v_logs["loss_risk"]
            v_steps += 1

    for k in v_sums: v_sums[k] /= max(v_steps, 1)
    demo_history["val_loss"].append(v_sums["loss"])
    demo_history["val_z"].append(v_sums["z"])
    demo_history["val_state"].append(v_sums["state"])
    demo_history["val_risk"].append(v_sums["risk"])

    elapsed = time.time() - t0
    print(f"[Epoch {epoch:02d}] train={sums['loss']:.4f} (z={sums['z']:.4f} state={sums['state']:.4f} risk={sums['risk']:.4f})  "
          f"val={v_sums['loss']:.4f}  ({elapsed:.1f}s)")

print(f"\nDone! Total time: {time.time() - t0:.1f}s")

### Demo Training Curve

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
epochs_x = range(EPOCHS)

# Total loss
axes[0].plot(epochs_x, demo_history["train_loss"], 'o-', label='Train', color='#1f77b4')
axes[0].plot(epochs_x, demo_history["val_loss"], 's-', label='Val', color='#ff7f0e')
axes[0].set_title('Total Rollout Loss')
axes[0].legend()

# Latent loss (z)
axes[1].plot(epochs_x, demo_history["train_z"], 'o-', label='Train', color='#1f77b4')
axes[1].plot(epochs_x, demo_history["val_z"], 's-', label='Val', color='#ff7f0e')
axes[1].set_title('Latent Loss (z)')
axes[1].legend()

# State loss
axes[2].plot(epochs_x, demo_history["train_state"], 'o-', label='Train', color='#1f77b4')
axes[2].plot(epochs_x, demo_history["val_state"], 's-', label='Val', color='#ff7f0e')
axes[2].set_title('State Loss (odom)')
axes[2].legend()

# Risk loss
axes[3].plot(epochs_x, demo_history["train_risk"], 'o-', label='Train', color='#1f77b4')
axes[3].plot(epochs_x, demo_history["val_risk"], 's-', label='Val', color='#ff7f0e')
axes[3].set_title('Risk Loss (collision)')
axes[3].legend()

for ax in axes:
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)

plt.suptitle('CNN + MLP Demo: Per-Component Loss Breakdown (5 epochs)', fontsize=13)
plt.tight_layout()
plt.show()

### What does the model predict after training?

Let's visualize a rollout on a validation episode: the model encodes frame 0, then predicts future states using only the actions. We compare the **predicted trajectory** (red) vs the **ground truth** (blue).

In [ ]:
model.eval()
n_vis = 4  # number of episodes to visualize
fig, axes = plt.subplots(2, n_vis, figsize=(5*n_vis, 10))

for idx in range(n_vis):
    sample = ds_val[idx]
    rgb_seq  = torch.from_numpy(sample["rgb"]).unsqueeze(0).to(device)
    depth_seq = torch.zeros(1, rgb_seq.shape[1], 1, 128, 128, device=device)
    odom_seq = torch.from_numpy(sample["odom"]).unsqueeze(0).to(device)
    imu_seq  = torch.from_numpy(sample["imu"]).unsqueeze(0).to(device)
    act_seq  = torch.from_numpy(sample["action"]).unsqueeze(0).to(device)

    T = rgb_seq.shape[1]
    H = min(16, T - 2)  # rollout horizon

    with torch.no_grad():
        # Encode frame 0
        out0 = model.encode(rgb_seq[0, 0:1], depth_seq[0, 0:1], odom_seq[0, 0:1], imu_seq[0, 0:1])
        z0 = out0.z

        # Rollout dynamics
        z_pred = model.rollout(z0, act_seq[:, :H])  # (1, H+1, z_dim)
        state_pred = model.state_head(z_pred)        # (1, H+1, 6)

    pred = state_pred[0].cpu().numpy()   # (H+1, 6)
    gt = odom_seq[0, :H+1, :6].cpu().numpy()

    # Top row: bird's-eye XY
    ax = axes[0, idx]
    ax.plot(gt[:, 0], gt[:, 1], 'b.-', label='Ground Truth', linewidth=2)
    ax.plot(pred[:, 0], pred[:, 1], 'r.--', label='Predicted', linewidth=2, alpha=0.8)
    ax.scatter(gt[0, 0], gt[0, 1], color='green', s=100, zorder=5, marker='*', label='Start')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.set_title(f'Episode {idx}: XY Trajectory')
    ax.legend(fontsize=7)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    # Bottom row: state dimensions over time
    ax = axes[1, idx]
    t_axis = np.arange(H+1)
    dim_labels = ['x', 'y', 'z', 'vx', 'vy', 'vz']
    for d in range(min(3, pred.shape[1])):
        ax.plot(t_axis, gt[:, d], color=f'C{d}', linestyle='-', linewidth=2, label=f'GT {dim_labels[d]}')
        ax.plot(t_axis, pred[:, d], color=f'C{d}', linestyle='--', linewidth=2, alpha=0.7, label=f'Pred {dim_labels[d]}')
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Value')
    ax.set_title(f'Episode {idx}: Position (solid=GT, dashed=pred)')
    ax.legend(fontsize=6, ncol=2)
    ax.grid(True, alpha=0.3)

plt.suptitle('Predicted vs Ground Truth Trajectories (Demo Model, 5 epochs)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Full Experiment Results (100-epoch SLURM Runs)

All 8 experiments were trained on A100 GPUs via SLURM (`scripts/world_model/train_world_model.py`).

| Experiment | Backbone | Dynamics | Dataset | Batch | Img | Train Time |
|------------|----------|----------|---------|-------|-----|------------|
| `cnn_mlp_c1` | CNN | MLP | collect1 | 64 | 128 | ~15 min |
| `cnn_rssm_c1` | CNN | RSSM | collect1 | 64 | 128 | ~15 min |
| `dino_mlp_c1` | DINOv2 | MLP | collect1 | 32 | 224 | ~70 min |
| `dino_rssm_c1` | DINOv2 | RSSM | collect1 | 32 | 224 | ~68 min |
| `cnn_mlp_c1u` | CNN | MLP | collect1_unfiltered | 64 | 128 | ~22 min |
| `cnn_rssm_c1u` | CNN | RSSM | collect1_unfiltered | 64 | 128 | ~22 min |
| `dino_mlp_c1u` | DINOv2 | MLP | collect1_unfiltered | 32 | 224 | ~113 min |
| `dino_rssm_c1u` | DINOv2 | RSSM | collect1_unfiltered | 32 | 224 | ~113 min |

### 4a. Load Training Curves

In [ ]:
# Parse epoch-level logs from SLURM output files
def parse_slurm_epochs(exp_name, search_dirs=None):
    """Parse [Epoch XXX] lines from SLURM .out files for a given experiment."""
    if search_dirs is None:
        search_dirs = [os.path.join(PROJECT_ROOT, "experiments")]
    train_loss, val_loss = {}, {}
    for d in search_dirs:
        pattern = os.path.join(d, f"{exp_name}_*.out")
        for fpath in sorted(glob.glob(pattern)):
            with open(fpath) as f:
                for line in f:
                    if line.startswith("[Epoch"):
                        parts = line.strip().split()
                        epoch = int(parts[1].rstrip("]"))
                        if "avg_loss=" in line:
                            train_loss[epoch] = float(line.split("avg_loss=")[1])
                        elif "val_loss=" in line:
                            val_loss[epoch] = float(line.split("val_loss=")[1])
    epochs = sorted(set(train_loss.keys()) & set(val_loss.keys()))
    return epochs, [train_loss[e] for e in epochs], [val_loss[e] for e in epochs]

# Also try loading pre-computed training curves JSON (from earlier v2 runs)
curves_path = os.path.join(PROJECT_ROOT, "experiments", "training_curves.json")
curves = {}
if os.path.exists(curves_path):
    with open(curves_path) as f:
        curves = json.load(f)

experiments = ["cnn_mlp_c1", "cnn_rssm_c1", "dino_mlp_c1", "dino_rssm_c1",
               "cnn_mlp_c1u", "cnn_rssm_c1u", "dino_mlp_c1u", "dino_rssm_c1u"]

full_curves = {}
for exp in experiments:
    epochs, train, val = parse_slurm_epochs(exp)
    # Merge with JSON curves (epochs 0-19) if SLURM logs are incomplete
    if exp in curves and len(epochs) < 50:
        json_train = [x[1] for x in curves[exp]["train"]]
        json_val = [x[1] for x in curves[exp]["val"]]
        merged_epochs = list(range(len(json_train)))
        for e, t, v in zip(epochs, train, val):
            if e >= len(json_train):
                merged_epochs.append(e)
                json_train.append(t)
                json_val.append(v)
        full_curves[exp] = {"epochs": merged_epochs, "train": json_train, "val": json_val}
    elif epochs:
        full_curves[exp] = {"epochs": epochs, "train": train, "val": val}
    else:
        print(f"  {exp}: no training logs found yet (job may still be running)")
        continue
    print(f"{exp}: {len(full_curves[exp]['epochs'])} epochs loaded")

### 4b. Training Curves — collect1 (filtered) vs collect1_unfiltered

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
colors = {"cnn_mlp": "#1f77b4", "cnn_rssm": "#ff7f0e", "dino_mlp": "#2ca02c", "dino_rssm": "#d62728"}
labels = {"cnn_mlp": "CNN+MLP", "cnn_rssm": "CNN+RSSM", "dino_mlp": "DINOv2+MLP", "dino_rssm": "DINOv2+RSSM"}

for col, (suffix, ds_label) in enumerate([("c1", "collect1 (filtered)"), ("c1u", "collect1_unfiltered")]):
    for prefix in ["cnn_mlp", "cnn_rssm", "dino_mlp", "dino_rssm"]:
        exp = f"{prefix}_{suffix}"
        if exp not in full_curves: continue
        c = full_curves[exp]
        axes[0, col].plot(c["epochs"], c["train"], color=colors[prefix], label=labels[prefix])
        axes[1, col].plot(c["epochs"], c["val"],   color=colors[prefix], label=labels[prefix])

    axes[0, col].set_title(f"Train Loss — {ds_label}")
    axes[1, col].set_title(f"Val Loss — {ds_label}")
    axes[0, col].set_ylabel("Loss")
    axes[1, col].set_ylabel("Loss")
    axes[1, col].set_xlabel("Epoch")
    axes[0, col].legend()
    axes[1, col].legend()
    for ax in axes[:, col]:
        ax.grid(True, alpha=0.3)
        ax.set_ylim(bottom=0)

plt.suptitle("Training Curves: All 8 Experiments (100 epochs)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 4b-extra. Overfitting Gap (Train vs Val)

In [ ]:
# Overfitting gap: val_loss - train_loss over time for each experiment
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for col, (suffix, ds_label) in enumerate([("c1", "collect1"), ("c1u", "collect1_unfiltered")]):
    ax = axes[col]
    for prefix in ["cnn_mlp", "cnn_rssm", "dino_mlp", "dino_rssm"]:
        exp = f"{prefix}_{suffix}"
        if exp not in full_curves: continue
        c = full_curves[exp]
        gap = [v - t for v, t in zip(c["val"], c["train"])]
        ax.plot(c["epochs"], gap, color=colors[prefix], label=labels[prefix], alpha=0.8)
    ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val Loss - Train Loss')
    ax.set_title(f'Overfitting Gap — {ds_label}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Overfitting Analysis: Positive = overfitting, Negative = underfitting', fontsize=13)
plt.tight_layout()
plt.show()

### 4c. Trajectory Evaluation (ADE / FDE)

After training, each model was evaluated on the validation set using **action-conditioned rollouts**:
- **ADE** (Average Displacement Error): mean L2 distance between predicted and ground-truth positions across all timesteps
- **FDE** (Final Displacement Error): L2 distance at the final predicted timestep
- Evaluated at 3 horizons: **1.6s** (8 steps), **3.2s** (16 steps), **5.0s** (25 steps)
- Broken down by **ground** vs **flight** episodes

In [ ]:
# Load trajectory evaluation results from all experiments
rows = []
for exp in experiments:
    eval_path = os.path.join(RESULTS_DIR, exp, "trajectory_eval.json")
    if not os.path.exists(eval_path):
        print(f"  [SKIP] {eval_path} not found")
        continue
    with open(eval_path) as f:
        data = json.load(f)
    for horizon_key, metrics in data.items():
        if horizon_key in ("experiment", "checkpoint"):
            continue
        rows.append({
            "Experiment": exp,
            "Horizon": horizon_key,
            "ADE": metrics["ade"],
            "FDE": metrics["fde"],
            "Ground ADE": metrics["ground_ade"],
            "Flight ADE": metrics["flight_ade"],
            "N Episodes": metrics["n_episodes"],
        })

df = pd.DataFrame(rows)

# Also load eval_metrics (pose MSE, accuracy, train time)
meta_rows = []
for exp in experiments:
    meta_path = os.path.join(RESULTS_DIR, exp, "eval_metrics.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            m = json.load(f)
        meta_rows.append({
            "Experiment": exp,
            "Pose MSE": m["pose_mse"],
            "Pose Acc": m["pose_accuracy"],
            "Train Time (min)": m["train_time_s"] / 60,
        })
df_meta = pd.DataFrame(meta_rows)

print("=== Evaluation Metrics ===")
display(df_meta.style.format({"Pose MSE": "{:.4f}", "Pose Acc": "{:.4f}", "Train Time (min)": "{:.1f}"}))

print("\n=== Trajectory ADE/FDE (all horizons) ===")
display(df.pivot_table(index="Experiment", columns="Horizon", values="ADE").round(4))

### 4d. ADE Comparison Plots

In [ ]:
# Grouped bar chart: ADE at 5.0s horizon for all 8 experiments
horizon = "5.0s"
df_h = df[df["Horizon"] == horizon].copy()

# Parse experiment name into components
df_h["Backbone"] = df_h["Experiment"].apply(lambda x: "DINOv2" if "dino" in x else "CNN")
df_h["Dynamics"] = df_h["Experiment"].apply(lambda x: "RSSM" if "rssm" in x else "MLP")
df_h["Dataset"]  = df_h["Experiment"].apply(lambda x: "unfiltered" if "c1u" in x else "filtered")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Overall ADE by experiment
ax = axes[0]
bars = ax.barh(df_h["Experiment"], df_h["ADE"], color=["#1f77b4" if "cnn" in e else "#2ca02c" for e in df_h["Experiment"]])
ax.set_xlabel("ADE (meters)")
ax.set_title(f"Overall ADE @ {horizon}")
ax.invert_yaxis()
for bar, val in zip(bars, df_h["ADE"]):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2, f"{val:.3f}", va='center', fontsize=9)

# Plot 2: Ground vs Flight ADE
ax = axes[1]
x = np.arange(len(df_h))
w = 0.35
ax.barh(x - w/2, df_h["Ground ADE"].values, w, label="Ground", color="#4CAF50", alpha=0.8)
ax.barh(x + w/2, df_h["Flight ADE"].values, w, label="Flight", color="#FF9800", alpha=0.8)
ax.set_yticks(x)
ax.set_yticklabels(df_h["Experiment"])
ax.set_xlabel("ADE (meters)")
ax.set_title(f"Ground vs Flight ADE @ {horizon}")
ax.legend()
ax.invert_yaxis()

# Plot 3: ADE across all horizons for top 4 (collect1_unfiltered)
ax = axes[2]
for exp in ["cnn_mlp_c1u", "cnn_rssm_c1u", "dino_mlp_c1u", "dino_rssm_c1u"]:
    subset = df[df["Experiment"] == exp].sort_values("Horizon")
    label = exp.replace("_c1u", "").replace("_", "+").upper()
    ax.plot(subset["Horizon"], subset["ADE"], "o-", label=label)
ax.set_xlabel("Horizon")
ax.set_ylabel("ADE (meters)")
ax.set_title("ADE vs Horizon (unfiltered dataset)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print best model
best = df_h.loc[df_h["ADE"].idxmin()]
print(f"\nBest overall ADE @ {horizon}: {best['Experiment']} = {best['ADE']:.4f}m")

### 4e. ADE Heatmap — All Experiments x All Horizons

In [ ]:
# Heatmap of ADE across all experiments and horizons
pivot_ade = df.pivot_table(index="Experiment", columns="Horizon", values="ADE")
pivot_fde = df.pivot_table(index="Experiment", columns="Horizon", values="FDE")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ADE heatmap
im1 = axes[0].imshow(pivot_ade.values, cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(len(pivot_ade.columns)))
axes[0].set_xticklabels(pivot_ade.columns)
axes[0].set_yticks(range(len(pivot_ade.index)))
axes[0].set_yticklabels(pivot_ade.index)
axes[0].set_title('ADE (lower = better)')
for i in range(len(pivot_ade.index)):
    for j in range(len(pivot_ade.columns)):
        val = pivot_ade.values[i, j]
        axes[0].text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=9,
                     color='white' if val > pivot_ade.values.mean() else 'black')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# FDE heatmap
im2 = axes[1].imshow(pivot_fde.values, cmap='YlOrRd', aspect='auto')
axes[1].set_xticks(range(len(pivot_fde.columns)))
axes[1].set_xticklabels(pivot_fde.columns)
axes[1].set_yticks(range(len(pivot_fde.index)))
axes[1].set_yticklabels(pivot_fde.index)
axes[1].set_title('FDE (lower = better)')
for i in range(len(pivot_fde.index)):
    for j in range(len(pivot_fde.columns)):
        val = pivot_fde.values[i, j]
        axes[1].text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=9,
                     color='white' if val > pivot_fde.values.mean() else 'black')
plt.colorbar(im2, ax=axes[1], shrink=0.8)

plt.suptitle('Trajectory Prediction Error Heatmaps (100-epoch models)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. MPC Inference with Trained Model

We load the best checkpoint and use **Cross-Entropy Method (CEM)** planning to find a velocity action sequence (lin_vel + ang_vel) that minimizes a cost function (distance to goal + risk penalty).

In [ ]:
# Load the best trained model from v3 (velocity actions, act_dim=6)
best_ckpt = os.path.join(RESULTS_DIR, "cnn_rssm_c1", "latest.pt")

if os.path.exists(best_ckpt):
    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
    mcfg = ckpt["model_config"]
    
    mpc_model = WorldModel(
        odom_dim=mcfg["odom_dim"], imu_dim=mcfg["imu_dim"], act_dim=mcfg["act_dim"],
        state_dim=mcfg["state_dim"], backbone="cnn", dynamics_type="rssm"
    ).to(device)
    mpc_model.load_state_dict(ckpt["model"])
    mpc_model.eval()
    print(f"Loaded checkpoint: epoch {ckpt['epoch']}, act_dim={mcfg['act_dim']}")
    
    # Get one batch for initial state encoding
    test_loader = DataLoader(ds_val, batch_size=1, shuffle=False, collate_fn=collate_hf)
    batch = next(iter(test_loader))
    
    with torch.no_grad():
        z = mpc_model.encode(
            batch.rgb[0, 0:1].to(device),
            batch.depth[0, 0:1].to(device),
            batch.odom[0, 0:1].to(device),
            batch.imu[0, 0:1].to(device)
        )
        z0 = z.z
    
    # CEM Planning — action_dim=6 (velocity)
    def cost_fn(state_seq, risk_seq, goal):
        dist = torch.norm(state_seq[:, -1, :2] - goal[:, :2], dim=-1)
        risk_penalty = risk_seq.mean(dim=1) * 100.0
        return dist + risk_penalty
    
    goal = torch.tensor([[5.0, 5.0]], device=device)
    cem_cfg = CEMConfig(horizon=15, iters=5, population=100, action_dim=mcfg["act_dim"])
    
    best_action, best_seq = cem_plan(mpc_model, z0, goal, cost_fn, cem_cfg, device)
    
    # Rollout the planned action sequence to get predicted states
    with torch.no_grad():
        z_planned = mpc_model.rollout(z0, best_seq.unsqueeze(0))
        states_planned = mpc_model.state_head(z_planned)[0].cpu().numpy()
    
    # --- Visualize the MPC Plan ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Bird's-eye view: start -> planned path -> goal
    ax = axes[0]
    start_pos = batch.odom[0, 0, :2].cpu().numpy()
    gt_traj = batch.odom[0, :, :2].cpu().numpy()
    ax.plot(gt_traj[:, 0], gt_traj[:, 1], 'b.-', alpha=0.4, label='GT trajectory')
    ax.plot(states_planned[:, 0], states_planned[:, 1], 'r.-', linewidth=2, markersize=8, label='MPC plan')
    ax.scatter(*start_pos, color='green', s=150, marker='*', zorder=5, label='Start')
    ax.scatter(5.0, 5.0, color='gold', s=150, marker='P', zorder=5, edgecolors='black', label='Goal')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.set_title('MPC Planned Trajectory (bird-eye)')
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_aspect('equal')
    
    # 2. Planned state dimensions over time
    ax = axes[1]
    for d, label in enumerate(['x', 'y', 'z']):
        ax.plot(states_planned[:, d], 'o-', label=label)
    ax.axhline(5.0, color='gold', linestyle='--', alpha=0.5, label='goal (x,y=5)')
    ax.set_xlabel('Timestep'); ax.set_ylabel('Position (m)')
    ax.set_title('Planned State Over Horizon')
    ax.legend(); ax.grid(True, alpha=0.3)
    
    # 3. Action heatmap — velocity components
    ax = axes[2]
    im = ax.imshow(best_seq.cpu().numpy().T, aspect='auto', cmap='RdBu_r')
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Action dim')
    vel_labels = ['vx', 'vy', 'vz', 'wx', 'wy', 'wz']
    ax.set_yticks(range(mcfg["act_dim"]))
    ax.set_yticklabels(vel_labels[:mcfg["act_dim"]])
    ax.set_title(f'Planned Velocity Actions ({best_seq.shape[0]} steps x {mcfg["act_dim"]}d)')
    plt.colorbar(im, ax=ax, shrink=0.8)
    
    plt.suptitle('CEM Model-Predictive Control: Goal-Directed Planning', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
    
    # Show the initial frame being planned from
    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(batch.rgb[0, 0].permute(1, 2, 0).cpu().numpy())
    ax.set_title("Input frame for MPC planning")
    ax.axis('off')
    plt.show()
    
else:
    print(f"Checkpoint not found: {best_ckpt}")
    print("v3 training jobs are still running. Check with: squeue -u $USER")

## 6. Cleanup

In [ ]:
import gc

for name in ['model', 'mpc_model', 'train_loader', 'val_loader', 'test_loader',
             'ds_train', 'ds_val', 'ds_hf']:
    if name in dir():
        exec(f"del {name}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Resources cleared.")